# 02 Composable Modules (DSPy, 2026)

## What This Lesson Is
Compose multiple modules into a staged pipeline with clear interfaces.

## Scientific Lens
- Concept: Modular decomposition for LLM workflows
- Measure: Stage-level pass rate and interface contract adherence
- Validity Limit: Small notebook pipelines do not expose long-chain propagation errors.


## How It Works
1. Compose deterministic stage pipeline.
2. Inspect stage outputs.
3. Run a live DSPy multi-stage pipeline.


In [ ]:
import os
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("Composable modules lesson preflight complete")


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
def normalize(text: str) -> str:
    return " ".join(text.lower().split())

def classify(summary: str) -> str:
    if "error" in summary:
        return "incident"
    if "deploy" in summary:
        return "release"
    return "general"

def format_result(label: str) -> dict:
    return {"label": label, "priority": "high" if label == "incident" else "normal"}

raw = " Deploy ERROR happened in production "
out = format_result(classify(normalize(raw)))
print(out)
assert out["label"] == "incident"


In [ ]:
# Live Demo
import os

try:
    import dspy
except Exception as exc:
    print(f"Skipping live DSPy module demo: dspy unavailable ({exc})")
else:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("Skipping live DSPy module demo: OPENAI_API_KEY not set.")
    else:
        dspy.configure(lm=dspy.LM("openai/gpt-4.1-mini", api_key=api_key, temperature=0))
        summarize = dspy.Predict("incident_text -> summary")
        classify = dspy.Predict("summary -> label")

        s = summarize(incident_text="Repeated auth failures caused login disruption for 15 minutes.")
        c = classify(summary=s.summary)
        print("summary:", s.summary)
        print("label:", c.label)
        assert s.summary and c.label


## Applied Labs
1. Add a remediation stage after classification and enforce stage contracts.
2. Inject malformed stage output and implement error isolation.
3. Measure per-stage failure rate across 20 live pipeline executions.

## Validation Checklist
- Each stage has a single clear responsibility.
- Intermediate outputs are inspectable and typed.
- Live pipeline produces both summary and classification outputs.

## Further Reading
- [DSPy Programming Guide](https://dspy.ai/learn/programming/)
- [Pipeline Composition Patterns](https://martinfowler.com/articles/collection-pipeline/)
- [Modular ML Systems](https://arxiv.org/abs/2205.02302)
